# HTDemucsWithSpatial Demo Site Preparation

This notebook:
1. Reads Reference, HTDemucs and HTDemucsWithSpatial datasets
2. Cut first 25s stems from chosen songs to be pasted into `docs/audio/`
3. Update song titles in the HTML
4. Allows to listen to all songs inline

**Dependences:** `pip install soundfile numpy ipython`
**ffmpeg** needs to be installed (`apt install ffmpeg` o `brew install ffmpeg`)

In [ ]:
# ── Installazione dipendenze (esegui solo la prima volta) ──
# !pip install soundfile numpy

## Configuration - modify here

In [1]:
from pathlib import Path

# ── Path to three datasets ──────────────────────────────────────────────────────
# Expected structure for each dataset:
#   <dataset_root>/<track_name>/drums.wav
#   <dataset_root>/<track_name>/bass.wav
#   <dataset_root>/<track_name>/vocals.wav
#   <dataset_root>/<track_name>/other.wav

_run_ts              = "20260414_170824"
REFERENCE_DIR        = Path(r"D:\Polimi\PhD\Dataset\binauralMUSDB18HQ\test")                # original stem
HTDEMUCS_DIR         = Path(r"D:\Polimi\PhD\Dataset\HTDemucs\binauralMUSDB18HQ\test")       # HT-Demucs output
HTDEMUCS_SPATIAL_DIR = Path(f"D:\Polimi\PhD\Dataset\SAHTDemucs\estimates_{_run_ts}")        # SA-HTDemucs output

# ── docs folder ──────────────────────────────────────────────────────────────────
DOCS_DIR = Path("../docs")   # es. Path("../docs") se sei dentro la repo

# ── Path to HTML file to update ──────────────────────────────────────────────────
HTML_FILE = DOCS_DIR / "index.html"

# ── Songs to include in the demo (exactly as they appear in the folders) ─────────
TRACKS = [
    {
        "folder_name": "The Easton Ellises - Falcon 69",
        "display_title": "The Easton Ellises — Falcon 69",
        "license": "CC BY-NC-SA",
        "slot": "song1",
    },
    {
        "folder_name": "Al James - Schoolboy Facination",   # folder name in the dataset
        "display_title": "Al James — Schoolboy Facination", # HTML song title
        "license": "CC BY-NC-SA",
        "slot": "song2",
    },
    {
        "folder_name": "Hollow Ground - Ill Fate",   # folder name in the dataset
        "display_title": "Hollow Ground — Ill Fate", # HTML song title
        "license": "CC BY-NC-SA",
        "slot": "song3",
    },
]

# ── Audio parameters ─────────────────────────────────────────────────────────────
CLIP_DURATION = 25      # seconds
CLIP_START    = 30      # start offset in seconds (skip intro)
STEMS         = ["drums", "bass", "vocals", "other"]

print("Loaded configuration.")
print(f"  Reference:            {REFERENCE_DIR}")
print(f"  HT-Demucs:            {HTDEMUCS_DIR}")
print(f"  SA-HTDemucs:          {HTDEMUCS_SPATIAL_DIR}")
print(f"  Docs:                 {DOCS_DIR}")

Loaded configuration.
  Reference:            D:\Polimi\PhD\Dataset\binauralMUSDB18HQ\test
  HT-Demucs:            D:\Polimi\PhD\Dataset\HTDemucs\binauralMUSDB18HQ\test
  SA-HTDemucs:          D:\Polimi\PhD\Dataset\SAHTDemucs\estimates_20260414_170824
  Docs:                 ..\docs


## Explore available tracks

In [2]:
def list_tracks(dataset_dir: Path, label: str):
    tracks = sorted([p.name for p in dataset_dir.iterdir() if p.is_dir()])
    print(f"\n📁 {label} ({len(tracks)} tracks):")
    for i, t in enumerate(tracks):
        print(f"  [{i:02d}] {t}")
    return tracks

ref_tracks = list_tracks(REFERENCE_DIR, "Reference")


📁 Reference (50 tracks):
  [00] AM Contra - Heart Peripheral
  [01] Al James - Schoolboy Facination
  [02] Angels In Amplifiers - I'm Alright
  [03] Arise - Run Run Run
  [04] BKS - Bulldozer
  [05] BKS - Too Much
  [06] Ben Carrigan - We'll Talk About It All Tonight
  [07] Bobby Nobody - Stitch Up
  [08] Buitraker - Revo X
  [09] Carlos Gonzalez - A Place For Us
  [10] Cristina Vane - So Easy
  [11] Detsky Sad - Walkie Talkie
  [12] Enda Reilly - Cur An Long Ag Seol
  [13] Forkupines - Semantics
  [14] Georgia Wonder - Siren
  [15] Girls Under Glass - We Feel Alright
  [16] Hollow Ground - Ill Fate
  [17] James Elder & Mark M Thompson - The English Actor
  [18] Juliet's Rescue - Heartbeats
  [19] Little Chicago's Finest - My Own
  [20] Louis Cressy Band - Good Time
  [21] Lyndsey Ollard - Catching Up
  [22] M.E.R.C. Music - Knockout
  [23] Moosmusic - Big Dummy Shake
  [24] Motor Tapes - Shore
  [25] Mu - Too Bright
  [26] Nerve 9 - Pray For The Rain
  [27] PR - Happy Daze
  [28] PR 

## Copy and cut stems into `docs/audio/`

In [3]:
import subprocess
import shutil

def ffmpeg_trim(src: Path, dst: Path, start: int, duration: int):
    """Cut WAV file with ffmpeg and save it into dst."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start),
        "-t",  str(duration),
        "-i",  str(src),
        "-c:a", "pcm_s16le",   # WAV 16-bit
        str(dst)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f" ⚠️ ffmpeg error su {src.name}:\n{result.stderr[-300:]}")
        return False
    return True


DATASETS = [
    (REFERENCE_DIR,        "ref"),
    (HTDEMUCS_DIR,         "htdemucs"),
    (HTDEMUCS_SPATIAL_DIR, "spatial"),
]

for track in TRACKS:
    slot       = track["slot"]
    folder     = track["folder_name"]
    out_dir    = DOCS_DIR / "audio" / slot
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n🎵 {folder} → docs/audio/{slot}/")

    for dataset_dir, prefix in DATASETS:
        track_path = dataset_dir / folder

        # mixture (only for the reference)
        if prefix == "ref":
            for mix_name in ["mixture.wav", "mix.wav"]:
                mix_src = track_path / mix_name
                if mix_src.exists():
                    ok = ffmpeg_trim(mix_src, out_dir / "mixture.wav", CLIP_START, CLIP_DURATION)
                    print(f"  {'✅' if ok else '❌'} mixture.wav")
                    break

        for stem in STEMS:
            src = track_path / f"{stem}.wav"
            dst = out_dir / f"{prefix}_{stem}.wav"
            if src.exists():
                ok = ffmpeg_trim(src, dst, CLIP_START, CLIP_DURATION)
                print(f"  {'✅' if ok else '❌'} {dst.name}")
            else:
                print(f"  ❌ Not found: {src}")

print("\n✅ Completed copy.")


🎵 The Easton Ellises - Falcon 69 → docs/audio/song1/
  ✅ mixture.wav
  ✅ ref_drums.wav
  ✅ ref_bass.wav
  ✅ ref_vocals.wav
  ✅ ref_other.wav
  ✅ htdemucs_drums.wav
  ✅ htdemucs_bass.wav
  ✅ htdemucs_vocals.wav
  ✅ htdemucs_other.wav
  ✅ spatial_drums.wav
  ✅ spatial_bass.wav
  ✅ spatial_vocals.wav
  ✅ spatial_other.wav

🎵 Al James - Schoolboy Facination → docs/audio/song2/
  ✅ mixture.wav
  ✅ ref_drums.wav
  ✅ ref_bass.wav
  ✅ ref_vocals.wav
  ✅ ref_other.wav
  ✅ htdemucs_drums.wav
  ✅ htdemucs_bass.wav
  ✅ htdemucs_vocals.wav
  ✅ htdemucs_other.wav
  ✅ spatial_drums.wav
  ✅ spatial_bass.wav
  ✅ spatial_vocals.wav
  ✅ spatial_other.wav

🎵 Hollow Ground - Ill Fate → docs/audio/song3/
  ✅ mixture.wav
  ✅ ref_drums.wav
  ✅ ref_bass.wav
  ✅ ref_vocals.wav
  ✅ ref_other.wav
  ✅ htdemucs_drums.wav
  ✅ htdemucs_bass.wav
  ✅ htdemucs_vocals.wav
  ✅ htdemucs_other.wav
  ✅ spatial_drums.wav
  ✅ spatial_bass.wav
  ✅ spatial_vocals.wav
  ✅ spatial_other.wav

✅ Completed copy.


## Update title into HTML

In [4]:
import re

html = HTML_FILE.read_text(encoding="utf-8")

for track in TRACKS:
    slot    = track["slot"]
    title   = track["display_title"]
    license_ = track["license"]

    # Update track-title in the correct block
    # find pattern into id="songN"
    pattern = (
        r'(id="' + slot + r'".*?<h3 class="track-title">)'
        r'(.*?)'
        r'(</h3>)'
    )
    html, n = re.subn(pattern, rf'\g<1>{title}\g<3>', html, count=1, flags=re.DOTALL)
    print(f"  {'✅' if n else '❌'} {slot} title → \"{title}\"")

    # Update license
    pattern_lic = (
        r'(id="' + slot + r'".*?<span class="track-license">)'
        r'(.*?)'
        r'(</span>)'
    )
    html, n = re.subn(pattern_lic, rf'\g<1>{license_}\g<3>', html, count=1, flags=re.DOTALL)
    print(f"  {'✅' if n else '❌'} {slot} license → \"{license_}\"")

HTML_FILE.write_text(html, encoding="utf-8")
print("\n✅ HTML updated.")

  ✅ song1 title → "The Easton Ellises — Falcon 69"
  ✅ song1 license → "CC BY-NC-SA"
  ✅ song2 title → "Al James — Schoolboy Facination"
  ✅ song2 license → "CC BY-NC-SA"
  ✅ song3 title → "Hollow Ground — Ill Fate"
  ✅ song3 license → "CC BY-NC-SA"

✅ HTML updated.


## Inline listening — Reference

In [ ]:
import IPython.display as ipd
from IPython.display import display, HTML

def play_stem(slot: str, prefix: str, stem: str):
    path = DOCS_DIR / "audio" / slot / f"{prefix}_{stem}.wav"
    if path.exists():
        display(HTML(f"<b>{prefix} / {stem}</b>"))
        display(ipd.Audio(str(path)))
    else:
        print(f"File non trovato: {path}")

def play_track(slot: str, prefix: str):
    track_title = next(t["display_title"] for t in TRACKS if t["slot"] == slot)
    display(HTML(f"<h3>🎵 {track_title} — {prefix}</h3>"))
    for stem in STEMS:
        play_stem(slot, prefix, stem)

# ── Ascolta la mixture ────────────────────────────────────────────────────────
for track in TRACKS:
    slot = track["slot"]
    display(HTML(f"<h3>🎵 {track['display_title']} — Mixture</h3>"))
    mix = DOCS_DIR / "audio" / slot / "mixture.wav"
    if mix.exists():
        display(ipd.Audio(str(mix)))

In [ ]:
# ── Reference ─────────────────────────────────────────────────────────────────
for track in TRACKS:
    play_track(track["slot"], "ref")

## Inline listening — HTDemucs

In [ ]:
for track in TRACKS:
    play_track(track["slot"], "htdemucs")

## Inline listening — HTDemucsWithSpatial

In [ ]:
for track in TRACKS:
    play_track(track["slot"], "spatial")

## Compare stem by stem (Reference vs HTDemucs vs HTDemucsWithSpatial)

In [ ]:
for track in TRACKS:
    slot = track["slot"]
    display(HTML(f"<h2>🎵 {track['display_title']}</h2>"))
    for stem in STEMS:
        display(HTML(f"<h4>── {stem.upper()} ──</h4>"))
        for prefix in ["ref", "htdemucs", "spatial"]:
            play_stem(slot, prefix, stem)

In [6]:
import subprocess, os

REPO_ROOT = DOCS_DIR.parent   # root della repo (un livello sopra docs/)

def git(cmd):
    # accetta sia stringa che lista
    if isinstance(cmd, str):
        cmd = cmd.split()
    result = subprocess.run(
        cmd, cwd=REPO_ROOT, capture_output=True, text=True
    )
    out = result.stdout.strip() or result.stderr.strip()
    if out:
        print(out)
    return result.returncode == 0

track_names = ", ".join(t["display_title"] for t in TRACKS)
commit_msg  = f"demo: update audio samples — {track_names}"

git("git add docs/")
git(["git", "commit", "-m", commit_msg])   # ← lista, non stringa
git("git push")

print("\n✅ Push completed. Site will update in ~60 seconds.")

[main f1996d3] demo: update audio samples â€” The Easton Ellises â€” Falcon 69, Al James â€” Schoolboy Facination
 9 files changed, 2 insertions(+), 2 deletions(-)
To https://github.com/Atzerbi/msslnet.git
   0d42a79..f1996d3  main -> main

✅ Push completed. Site will update in ~60 seconds.
